# Scrape & revise jobs

Scrape postings from a board, then clean and label each description with the
LLM. Only the final labeled result is saved, to
`data/jobs_<timestamp>_revised.jsonl` (no intermediate raw file).

Edit the settings cell, then run the cells top to bottom.


## Setup: make the project's `src/` importable

Our code lives in `src/`, so we add it to the import path. Run this once.


In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd()
# If you launched from the notebooks/ folder, step up to the project root.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from openai import OpenAI

from scraper import DATA_DIR, ScrapeConfig, scrape
from revise import LLMSettings, revise, save_revised

print(f"Project root: {ROOT}")
print(f"Output goes to: {DATA_DIR}")

Project root: /home/yokesh/Projects/job-ai
Output goes to: /home/yokesh/Projects/job-ai/data


## Settings — edit these, then run the next cell

Give the **job title** and the **board** (website) to scrape.
`linkedin_fetch_description=True` is important: without it the postings come
back with no description text.


In [4]:
JOB_TITLE = "AI Engineering Manager"   # the search term
BOARD = "linkedin"                      # one of: linkedin indeed glassdoor google
LOCATION = "Canada"
RESULTS_WANTED = 10                     # how many postings to pull
HOURS_OLD = 168                         # only postings newer than this many hours (168 = 1 week)
IS_REMOTE = False

config = ScrapeConfig(
    site_name=[BOARD],
    search_term=JOB_TITLE,
    location=LOCATION,
    results_wanted=RESULTS_WANTED,
    hours_old=HOURS_OLD,
    is_remote=IS_REMOTE,
    country_indeed="canada",            # only used by indeed/glassdoor
    linkedin_fetch_description=True,    # pull the full JD text (needed for later steps)
)
config

ScrapeConfig(site_name=[<Site.linkedin: 'linkedin'>], search_term='AI Engineering Manager', location='Canada', google_search_term=None, results_wanted=10, distance=50, job_type=None, is_remote=False, hours_old=168, country_indeed='canada', linkedin_fetch_description=True, verbose=1)

## Scrape

This calls the job board live, so it takes a few seconds and needs internet.


In [5]:
jobs = scrape(config)
print(f"Scraped {len(jobs)} jobs")


2026-06-20 14:02:06,111 - INFO - JobSpy:Linkedin - finished scraping


Scraped 10 jobs


In [6]:
for i, job in enumerate(jobs, 1):
  print(f"{i}. {job.title} at {job.company} ({job.location})")
  print(f"   URL: {job.job_url}")
  print()

1. Engineering Manager, Growth at Jobgether ()
   URL: https://www.linkedin.com/jobs/view/4429668080

2. Engineering Manager at Badge (Vancouver, British Columbia, Canada)
   URL: https://www.linkedin.com/jobs/view/4401550595

3. Software Engineering Manager at PointClickCare (Mississauga, Ontario, Canada)
   URL: https://www.linkedin.com/jobs/view/4409993396

4. Manager, Applied AI/ML, Data Science & Engineering at Jobgether ()
   URL: https://www.linkedin.com/jobs/view/4429308527

5. Manager, Software Engineering at AltaML (Calgary, Alberta, Canada)
   URL: https://www.linkedin.com/jobs/view/4430732998

6. Engineering Manager - Data Platform at Two Circles (Vancouver, British Columbia, Canada)
   URL: https://www.linkedin.com/jobs/view/4427866223

7. Engineering Manager at IMPACT Pump Solutions (Calgary, Alberta, Canada)
   URL: https://www.linkedin.com/jobs/view/4430132712

8. Global Engineering Manager-Manufacturing at Gowan Company (Calgary, Alberta, Canada)
   URL: https://www.li

## Peek at what we got

A quick table of the titles, companies, and whether a description came back.


In [7]:
import pandas as pd

preview = pd.DataFrame(
    {
        "title": [j.title for j in jobs],
        "company": [j.company for j in jobs],
        "location": [j.location for j in jobs],
        "has_description": [bool(j.description) for j in jobs],
    }
)
preview

,title,company,location,has_description
0,"Engineering Manager, Growth",Jobgether,,True
1,Engineering Manager,Badge,"Vancouver, British Columbia, Canada",True
2,Software Engineering Manager,PointClickCare,"Mississauga, Ontario, Canada",True
3,"Manager, Applied AI/ML, Data Science & Enginee...",Jobgether,,True
4,"Manager, Software Engineering",AltaML,"Calgary, Alberta, Canada",True
5,Engineering Manager - Data Platform,Two Circles,"Vancouver, British Columbia, Canada",True
6,Engineering Manager,IMPACT Pump Solutions,"Calgary, Alberta, Canada",True
7,Global Engineering Manager-Manufacturing,Gowan Company,"Calgary, Alberta, Canada",True
8,"Enterprise Solutions Engineering Manager, Canada",Verkada,"Toronto, Ontario, Canada",True
9,AI Engineering Manager,Anju Software,,True


## Revise — clean & label the descriptions

Send the scraped jobs to the LLM (credentials read from `.env`). It marks
section boundaries and labels each sentence (`requirement`, `responsibility`,
`preferred`, `context`) — the JD text itself is never reworded. Makes live API
calls, so it costs a little and takes a few seconds.

This writes **only** the final `data/jobs_<timestamp>_revised.jsonl`.


In [8]:
from datetime import datetime

settings = LLMSettings()  # reads OPENROUTER_* keys from .env
client = OpenAI(api_key=settings.api_key, base_url=settings.base_url)
print(f"Using model: {settings.model}")

revised = revise(jobs, client, settings.model)

# Name the output by timestamp; save_revised writes the "_revised.jsonl" only,
# so no intermediate raw file is created.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
revised_path = save_revised(revised, DATA_DIR / f"jobs_{stamp}.jsonl")
print(f"Saved {len(revised)} revised jobs -> {revised_path.relative_to(ROOT)}")

Using model: anthropic/claude-haiku-4.5
  batch 1/1 (10 jobs) — 2 LLM calls
    Engineering Manager, Growth — 3/5 sections kept, 27 chunks
    Engineering Manager — 3/4 sections kept, 21 chunks
    Software Engineering Manager — 3/7 sections kept, 17 chunks
    Manager, Applied AI/ML, Data Science & Engineering — 3/5 sections kept, 33 chunks
    Manager, Software Engineering — 3/7 sections kept, 30 chunks
    Engineering Manager - Data Platform — 3/4 sections kept, 50 chunks
    Engineering Manager — 3/5 sections kept, 40 chunks
    Global Engineering Manager-Manufacturing — 3/4 sections kept, 31 chunks
    Enterprise Solutions Engineering Manager, Canada — 3/7 sections kept, 16 chunks
    AI Engineering Manager — 3/4 sections kept, 46 chunks
Saved 10 revised jobs -> data/jobs_20260620_140234_revised.jsonl


## Inspect the labeled chunks

For the first job, show each JD sentence with the label the LLM assigned.


In [9]:
first = revised[0]
print(f"{first.title} — {first.company}")
print(f"{len(first.description_chunks)} chunks")

pd.DataFrame(
    {
        "label": [chunk.label.value for chunk in first.description_chunks],
        "section": [chunk.section for chunk in first.description_chunks],
        "text": [chunk.text for chunk in first.description_chunks],
    }
)

Engineering Manager, Growth — Jobgether
27 chunks


,label,section,text
0,context,Role summary,This position is listed on behalf of a partner...
1,context,Role summary,Our partner is looking for an Engineering Mana...
2,context,Role summary,This role sits at the intersection of product ...
3,responsibility,Role summary,"You will lead a high-performing, globally dist..."
4,responsibility,Role summary,The role combines people leadership with stron...
5,responsibility,Role summary,"You will work closely with Product, Design, Da..."
6,responsibility,Role summary,A key part of the role involves building a cul...
7,context,Role summary,This is a high-impact leadership position for ...
8,responsibility,Accountabilities,"Lead, hire, and develop a high-performing Grow..."
9,responsibility,Accountabilities,"Partner with Product, Design, Data, and Engine..."


In [10]:
import sys
from pathlib import Path
from datetime import datetime

_src = Path.cwd()
if _src.name == "notebooks":
    _src = _src.parent
_src = _src / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from openai import OpenAI
from scraper import DATA_DIR
from revise import LLMSettings, revise, save_revised

assert "jobs" in dir() and jobs, "No `jobs` in the kernel — run the scrape cells first."
print(f"{len(jobs)} jobs in memory; starting revise...")

settings = LLMSettings()
client = OpenAI(api_key=settings.api_key, base_url=settings.base_url)
print(f"Using model: {settings.model}")

revised = revise(jobs, client, settings.model)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
revised_path = save_revised(revised, DATA_DIR / f"jobs_{stamp}.jsonl")
print(f"\nSaved {len(revised)} revised jobs -> {revised_path}")

10 jobs in memory; starting revise...
Using model: anthropic/claude-haiku-4.5
  batch 1/1 (10 jobs) — 2 LLM calls
    Engineering Manager, Growth — 3/5 sections kept, 27 chunks
    Engineering Manager — 3/4 sections kept, 21 chunks
    Software Engineering Manager — 3/7 sections kept, 17 chunks
    Manager, Applied AI/ML, Data Science & Engineering — 3/5 sections kept, 33 chunks
    Manager, Software Engineering — 3/7 sections kept, 30 chunks
    Engineering Manager - Data Platform — 3/4 sections kept, 50 chunks
    Engineering Manager — 3/5 sections kept, 40 chunks
    Global Engineering Manager-Manufacturing — 3/4 sections kept, 31 chunks
    Enterprise Solutions Engineering Manager, Canada — 3/7 sections kept, 16 chunks
    AI Engineering Manager — 3/4 sections kept, 46 chunks

Saved 10 revised jobs -> /home/yokesh/Projects/job-ai/data/jobs_20260620_140302_revised.jsonl
